In [1]:
%%capture
# Unsloth + TRL (SFT) + deps — xem https://unsloth.ai/docs
!pip install -q unsloth
!pip install -q "transformers>=4.46.0" datasets trl peft accelerate bitsandbytes pandas matplotlib sacrebleu pyvi

In [2]:
# ====== CONFIG (Google Colab + Unsloth + Qwen2.5) ======
import glob
import os

# Model: ban 4-bit tren org unsloth (tiet kiem VRAM). Co the doi thanh "Qwen/Qwen2.5-7B-Instruct" + LOAD_IN_4BIT=True
MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
LOAD_IN_4BIT = True

# Mac dinh: dat file tai /content/finetune_qwen257B.jsonl (upload bang Files panel hoac cell upload ben duoi)
DEFAULT_DATA = '/content/finetune_qwen257B.jsonl'
found = glob.glob('/content/**/finetune_qwen257B.jsonl', recursive=True)
if os.path.isfile(DEFAULT_DATA):
    DATASET_PATH = DEFAULT_DATA
elif found:
    DATASET_PATH = found[0]
else:
    DATASET_PATH = DEFAULT_DATA  # se bao loi neu chua upload — sua tay sau khi upload

OUTPUT_DIR = '/content/qwen25_7b_instruct_lora'

MAX_LENGTH = 2048
NUM_TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
SAVE_STEPS = 100
LOGGING_STEPS = 10

# Split data for early stopping (val) + final test report (test)
VAL_SIZE = 0.05
TEST_SIZE = 0.05
EVAL_STEPS = SAVE_STEPS
EARLY_STOPPING_PATIENCE = 7

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
MERGE_AND_SAVE = False

print('MODEL_NAME   =', MODEL_NAME)
print('DATASET_PATH =', DATASET_PATH)
print('OUTPUT_DIR   =', OUTPUT_DIR)

MODEL_NAME   = unsloth/Qwen2.5-7B-Instruct-bnb-4bit
DATASET_PATH = /content/finetune_qwen257B.jsonl
OUTPUT_DIR   = /content/qwen25_7b_instruct_lora


In [3]:
import json
from pathlib import Path
import torch

def load_records(path: str):
    fp = Path(path)
    if not fp.exists():
        raise FileNotFoundError(f'Dataset not found: {path}')

    if fp.suffix.lower() == '.csv':
        import pandas as pd
        return pd.read_csv(fp).to_dict(orient='records')
    if fp.suffix.lower() == '.json':
        data = json.loads(fp.read_text(encoding='utf-8'))
        if isinstance(data, dict) and 'data' in data and isinstance(data['data'], list):
            return data['data']
        if isinstance(data, list):
            return data
        raise ValueError("JSON format khong hop le (can list hoac {'data': list})")
    if fp.suffix.lower() == '.jsonl':
        rows = []
        with fp.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows
    raise ValueError('Unsupported dataset format. Use csv/json/jsonl.')


def row_to_messages(row):
    """Chuyen sample (instruction/input/output) -> chat messages cho Qwen2.5."""
    instruction = str(row.get('instruction', row.get('question', row.get('prompt', '')))).strip()
    input_text = str(row.get('input', '')).strip()
    output = str(row.get('output', row.get('answer', row.get('response', '')))).strip()

    if not instruction:
        raise ValueError('Missing instruction/question/prompt')
    if not output:
        raise ValueError('Missing output/answer/response')

    user_content = f'{instruction}\n\n{input_text}' if input_text else instruction
    return [
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': output},
    ]

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Torch: 2.10.0+cu128
CUDA available: True


In [4]:
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.backends.cuda.matmul.allow_tf32 = True

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.15 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [5]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import EarlyStoppingCallback, TrainingArguments
from unsloth import is_bfloat16_supported
import random


def build_sft_dataset(dataset_path: str, tokenizer):
    raw = load_records(dataset_path)
    samples = []
    for r in raw:
        msgs = row_to_messages(r)
        user_msg = msgs[0]
        assistant_msg = msgs[1]

        # Text huấn luyện (bao gồm cả phần assistant trả lời đúng chuẩn)
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )

        # Prompt chỉ chứa user + add_generation_prompt để generate câu trả lời
        prompt = tokenizer.apply_chat_template(
            [user_msg],
            tokenize=False,
            add_generation_prompt=True,
        )

        # Keep faq_id để split theo nhóm (tránh data leakage)
        samples.append({
            'text': text,
            'prompt': prompt,
            'gold': assistant_msg.get('content', '').strip(),
            'faq_id': r.get('faq_id', None),
        })
    return Dataset.from_list(samples)


full_ds = build_sft_dataset(DATASET_PATH, tokenizer)
print('Total samples:', len(full_ds))

use_eval = (VAL_SIZE + TEST_SIZE) > 0

def split_by_group(ds, group_field: str, val_frac: float, test_frac: float, seed: int = 3407):
    total = len(ds)
    target_val = int(total * val_frac)
    target_test = int(total * test_frac)

    group_to_indices = {}
    # ds[group_field] is a python list (fast enough for ~11k+ rows).
    for idx, gid in enumerate(ds[group_field]):
        group_to_indices.setdefault(gid, []).append(idx)

    group_ids = list(group_to_indices.keys())
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    train_indices = []
    val_indices = []
    test_indices = []
    val_count = 0
    test_count = 0

    for gid in group_ids:
        idxs = group_to_indices[gid]
        if val_count < target_val:
            val_indices.extend(idxs)
            val_count += len(idxs)
        elif test_count < target_test:
            test_indices.extend(idxs)
            test_count += len(idxs)
        else:
            train_indices.extend(idxs)

    # Sanity check: all rows assigned exactly once.
    assert len(train_indices) + len(val_indices) + len(test_indices) == total

    return (
        ds.select(train_indices),
        ds.select(val_indices),
        ds.select(test_indices),
    )


if use_eval:
    train_ds, val_ds, test_ds = split_by_group(
        full_ds,
        group_field='faq_id',
        val_frac=VAL_SIZE,
        test_frac=TEST_SIZE,
        seed=3407,
    )
    # If user turns off one split, keep the variable as None.
    if VAL_SIZE <= 0:
        val_ds = None
    if TEST_SIZE <= 0:
        test_ds = None
else:
    train_ds = full_ds
    val_ds = None
    test_ds = None

# Remove faq_id from datasets passed to the trainer.
if 'faq_id' in train_ds.column_names:
    train_ds = train_ds.remove_columns(['faq_id'])
if val_ds is not None and 'faq_id' in val_ds.column_names:
    val_ds = val_ds.remove_columns(['faq_id'])
if test_ds is not None and 'faq_id' in test_ds.column_names:
    test_ds = test_ds.remove_columns(['faq_id'])

print('Train samples:', len(train_ds))
print('Val samples:', len(val_ds) if val_ds is not None else 0)
print('Test samples:', len(test_ds) if test_ds is not None else 0)

import math

# Uoc tinh buoc optimizer / epoch (1 GPU) — gan voi cach Trainer dem batch + grad accum
_batches_per_epoch = math.ceil(len(train_ds) / TRAIN_BATCH_SIZE)
_update_steps_per_epoch = max(1, math.ceil(_batches_per_epoch / GRAD_ACCUM_STEPS))
_est_total_optimizer_steps = NUM_TRAIN_EPOCHS * _update_steps_per_epoch
WARMUP_STEPS = max(1, int(_est_total_optimizer_steps * WARMUP_RATIO))
print(
    'Estimated optimizer steps:', _est_total_optimizer_steps,
    '| warmup_steps:', WARMUP_STEPS,
    '(WARMUP_RATIO=', WARMUP_RATIO, ')',
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)] if val_ds is not None else []

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_strategy='steps' if val_ds is not None else 'no',
        eval_steps=EVAL_STEPS if val_ds is not None else None,
        load_best_model_at_end=val_ds is not None,
        metric_for_best_model='eval_loss' if val_ds is not None else None,
        greater_is_better=False,
        save_strategy='steps',
        save_total_limit=2,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        seed=3407,
        report_to='none',
        disable_tqdm=True,
        remove_unused_columns=False,
    ),
    callbacks=callbacks,
)

train_result = trainer.train()
print('Train result metrics:', train_result.metrics)

test_metrics = None

Total samples: 11650
Train samples: 10472
Val samples: 587
Test samples: 591
Estimated optimizer steps: 2618 | warmup_steps: 78 (WARMUP_RATIO= 0.03 )


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10472 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/587 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,472 | Num Epochs = 2 | Total steps = 2,618
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


{'loss': '4.119', 'grad_norm': '2.481', 'learning_rate': '2.308e-05', 'epoch': '0.007639'}
{'loss': '3.196', 'grad_norm': '1.528', 'learning_rate': '4.872e-05', 'epoch': '0.01528'}
{'loss': '2.204', 'grad_norm': '2.06', 'learning_rate': '7.436e-05', 'epoch': '0.02292'}
{'loss': '1.464', 'grad_norm': '0.714', 'learning_rate': '0.0001', 'epoch': '0.03056'}
{'loss': '1.222', 'grad_norm': '0.7826', 'learning_rate': '0.0001256', 'epoch': '0.0382'}
{'loss': '1.217', 'grad_norm': '0.6907', 'learning_rate': '0.0001513', 'epoch': '0.04584'}
{'loss': '1.1', 'grad_norm': '0.7702', 'learning_rate': '0.0001769', 'epoch': '0.05348'}
{'loss': '1.059', 'grad_norm': '0.8067', 'learning_rate': '0.0002', 'epoch': '0.06112'}
{'loss': '1.11', 'grad_norm': '0.7698', 'learning_rate': '0.0002', 'epoch': '0.06875'}


...

{'eval_loss': '1.065', 'eval_runtime': '107.9', 'eval_samples_per_second': '5.442', 'eval_steps_per_second': '2.726', 'epoch': '0.6112'}
{'loss': '0.5574', 'grad_norm': '1.035', 'learning_rate': '0.0001618', 'epoch': '0.6188'}
{'loss': '0.5254', 'grad_norm': '0.8936', 'learning_rate': '0.0001609', 'epoch': '0.6264'}
{'loss': '0.5482', 'grad_norm': '0.9775', 'learning_rate': '0.0001599', 'epoch': '0.6341'}
{'loss': '0.5103', 'grad_norm': '0.7356', 'learning_rate': '0.0001589', 'epoch': '0.6417'}
{'loss': '0.5371', 'grad_norm': '0.9202', 'learning_rate': '0.0001579', 'epoch': '0.6494'}
{'loss': '0.5047', 'grad_norm': '0.7338', 'learning_rate': '0.0001569', 'epoch': '0.657'}
{'loss': '0.5099', 'grad_norm': '0.6925', 'learning_rate': '0.0001558', 'epoch': '0.6646'}
{'loss': '0.5602', 'grad_norm': '0.6372', 'learning_rate': '0.0001548', 'epoch': '0.6723'}
{'loss': '0.5018', 'grad_norm': '0.7662', 'learning_rate': '0.0001538', 'epoch': '0.6799'}
{'loss': '0.5569', 'grad_norm': '0.6922', 'lea

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

{'eval_loss': '1.07', 'eval_runtime': '107.3', 'eval_samples_per_second': '5.47', 'eval_steps_per_second': '2.74', 'epoch': '0.6875'}
{'loss': '0.553', 'grad_norm': '0.7739', 'learning_rate': '0.0001517', 'epoch': '0.6952'}
{'loss': '0.5349', 'grad_norm': '0.8108', 'learning_rate': '0.0001506', 'epoch': '0.7028'}
{'loss': '0.5274', 'grad_norm': '0.7614', 'learning_rate': '0.0001495', 'epoch': '0.7105'}
{'loss': '0.5384', 'grad_norm': '0.9635', 'learning_rate': '0.0001485', 'epoch': '0.7181'}
{'loss': '0.5258', 'grad_norm': '0.5618', 'learning_rate': '0.0001474', 'epoch': '0.7257'}
{'loss': '0.5548', 'grad_norm': '0.7478', 'learning_rate': '0.0001463', 'epoch': '0.7334'}
{'loss': '0.5372', 'grad_norm': '0.8492', 'learning_rate': '0.0001452', 'epoch': '0.741'}
{'loss': '0.5169', 'grad_norm': '0.7579', 'learning_rate': '0.0001441', 'epoch': '0.7487'}
{'loss': '0.5275', 'grad_norm': '0.5806', 'learning_rate': '0.000143', 'epoch': '0.7563'}
{'loss': '0.5338', 'grad_norm': '0.5761', 'learnin

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

{'eval_loss': '1.078', 'eval_runtime': '106.9', 'eval_samples_per_second': '5.49', 'eval_steps_per_second': '2.749', 'epoch': '0.7639'}
{'train_runtime': '4194', 'train_samples_per_second': '4.993', 'train_steps_per_second': '0.624', 'train_loss': '0.7889', 'epoch': '0.7639'}
Train result metrics: {'train_runtime': 4194.4349, 'train_samples_per_second': 4.993, 'train_steps_per_second': 0.624, 'train_loss': 0.7889169549942017, 'epoch': 0.7639419404125286}


In [6]:
# --- Test evaluation (run in a separate cell) ---

test_metrics = None

# Guard: make sure you ran the train cell first.
if 'trainer' not in globals():
    raise RuntimeError('trainer is not defined. Run the train cell first.')

if getattr(trainer.state, 'global_step', 0) == 0:
    print('[WARN] global_step == 0. Run trainer.train() before evaluating test.')

if test_ds is not None and getattr(trainer.state, 'global_step', 0) > 0:
    try:
        # Tokenize the raw `text` column to make sure collator can find `input_ids`.
        test_eval_ds = test_ds.map(
            lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH),
            batched=True,
            remove_columns=['text'],
            desc='Tokenizing test set (post-train)',
        )

        # Quick sanity check (first row) before evaluation.
        _row0 = test_eval_ds[0]
        if 'input_ids' not in _row0:
            raise KeyError("test_eval_ds missing 'input_ids' after tokenization")

        # Workaround: Transformers' NotebookProgressCallback expects on_train_begin
        # before on_evaluate. For standalone eval we remove that callback.
        try:
            from transformers.utils.notebook import NotebookProgressCallback

            if hasattr(trainer, 'callback_handler') and hasattr(trainer.callback_handler, 'callbacks'):
                trainer.callback_handler.callbacks = [
                    cb for cb in trainer.callback_handler.callbacks
                    if not isinstance(cb, NotebookProgressCallback)
                ]
        except Exception:
            pass

        test_metrics = trainer.evaluate(eval_dataset=test_eval_ds)
        print('Test metrics:', test_metrics)

        # --------------------
        # BLEU evaluation (generate) on test split
        # --------------------
        from sacrebleu.metrics import BLEU as SacreBLEU
        from pyvi import ViTokenizer
        import numpy as np

        BLEU_MAX_SAMPLES = 200  # tránh generate quá lâu
        MAX_NEW_TOKENS = 256
        MAX_NEW_TOKENS = int(MAX_NEW_TOKENS)

        bleu = SacreBLEU(tokenize='none', effective_order=True)

        n = len(test_ds)
        n_use = min(n, BLEU_MAX_SAMPLES)

        def detokenize_vn(tokenized: str) -> str:
            # ViTokenizer.tokenize() tách dấu câu bằng khoảng trắng (vd: " .")
            # => gom lại trước khi đưa sang sacrebleu (tokenize='none').
            if tokenized is None:
                return ""
            s = str(tokenized)
            s = " ".join(s.split())
            s = s.replace(" .", ".")
            s = s.replace(" ,", ",")
            s = s.replace(" !", "!")
            s = s.replace(" ?", "?")
            s = s.replace(" ;", ";")
            s = s.replace(" :", ":")
            s = s.replace(" )", ")")
            s = s.replace("( ", "(")
            return s

        sys_stream = []
        ref_stream = []

        # Dùng greedy decoding để ổn định đánh giá
        model.eval()
        with torch.no_grad():
            for i in range(n_use):
                sample = test_ds[i]
                prompt = sample.get('prompt', '')
                gold = sample.get('gold', '')

                if not prompt or not gold:
                    continue

                model_inputs = tokenizer(
                    prompt,
                    return_tensors='pt',
                    truncation=True,
                    max_length=MAX_LENGTH,
                ).to(next(model.parameters()).device)
                input_len = model_inputs['input_ids'].shape[1]

                gen_ids = model.generate(
                    **model_inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    temperature=0.0,
                    pad_token_id=tokenizer.eos_token_id,
                    use_cache=False,  # tránh lỗi RoPE fast kernel / attention mask shape mismatch
                )

                gen_tokens = gen_ids[0, input_len:]
                gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

                # Tokenize tiếng Việt rồi detokenize lại để sacrebleu không cảnh báo
                gen_tok = ViTokenizer.tokenize(gen_text)
                gold_tok = ViTokenizer.tokenize(gold)

                gen_detok = detokenize_vn(gen_tok)
                gold_detok = detokenize_vn(gold_tok)

                sys_stream.append(gen_detok)
                ref_stream.append(gold_detok)

        # sacrebleu BLEU corpus: ref_stream là list[str] => truyền dưới dạng [ref_streams]
        if len(sys_stream) > 0:
            bleu_score = bleu.corpus_score(sys_stream, [ref_stream]).score
        else:
            bleu_score = None

        print(f"Corpus BLEU-4 (detokenized Vietnamese): {bleu_score}")
        if isinstance(test_metrics, dict):
            test_metrics['bleu_corpus4'] = bleu_score
        else:
            test_metrics = {'eval': test_metrics, 'bleu_corpus4': bleu_score}
        print('Test metrics (after BLEU):', test_metrics)
    except RuntimeError as e:
        # This can happen if the notebook progress/callback state is inconsistent.
        print('[ERROR] Test evaluation failed:', str(e))
        print('Hint: restart kernel and run cells in order: train cell -> this test cell.')
else:
    print('Test ds is None (TEST_SIZE=0) or skipped because train was not run.')


Tokenizing test set (post-train):   0%|          | 0/591 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

{'eval_loss': '0.9883', 'eval_runtime': '102.6', 'eval_samples_per_second': '5.761', 'eval_steps_per_second': '2.885', 'epoch': '0.7639'}
Test metrics: {'eval_loss': 0.9882776141166687, 'eval_runtime': 102.584, 'eval_samples_per_second': 5.761, 'eval_steps_per_second': 2.885, 'epoch': 0.7639419404125286}


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Corpus BLEU-4 (detokenized Vietnamese): 4.885318098436546
Test metrics (after BLEU): {'eval_loss': 0.9882776141166687, 'eval_runtime': 102.584, 'eval_samples_per_second': 5.761, 'eval_steps_per_second': 2.885, 'epoch': 0.7639419404125286, 'bleu_corpus4': 4.885318098436546}


In [7]:
# --- Bieu do + xuat file phuc vu bao cao (chay ngay sau cell train) ---
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

REPORT_DIR = os.path.join(OUTPUT_DIR, 'training_report')
os.makedirs(REPORT_DIR, exist_ok=True)

log_history = list(trainer.state.log_history)

# Luu log day du (Excel / phan tich them)
pd.DataFrame(log_history).to_csv(os.path.join(REPORT_DIR, 'training_log.csv'), index=False, encoding='utf-8-sig')

train_loss_pts = [(e['step'], e['loss']) for e in log_history if 'loss' in e and e.get('loss') is not None]
eval_loss_pts = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e and e.get('eval_loss') is not None]
lr_pts = [(e['step'], e['learning_rate']) for e in log_history if 'learning_rate' in e]


def save_fig(fig, filename: str) -> str:
    path = os.path.join(REPORT_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path

# 1) Loss train + val
fig1, ax = plt.subplots(figsize=(9, 5))
if train_loss_pts:
    xs, ys = zip(*train_loss_pts)
    ax.plot(xs, ys, label='Train loss', color='#2563eb', linewidth=1.2)
if eval_loss_pts:
    xs, ys = zip(*eval_loss_pts)
    ax.plot(xs, ys, label='Validation loss', color='#ea580c', linewidth=1.2, marker='o', markersize=3)
ax.set_xlabel('Step (global step)')
ax.set_ylabel('Loss')
ax.set_title('Qua trinh huan luyen: loss train / validation')
ax.legend()
ax.grid(True, alpha=0.35)
fig1.tight_layout()
p1 = save_fig(fig1, 'loss_curve.png')
print('Saved:', p1)

# 2) Learning rate
if lr_pts:
    fig2, ax2 = plt.subplots(figsize=(9, 4))
    xs, ys = zip(*lr_pts)
    ax2.plot(xs, ys, color='#16a34a', linewidth=1.2)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Learning rate')
    ax2.set_title('Lich trinh learning rate (warmup + cosine)')
    ax2.grid(True, alpha=0.35)
    fig2.tight_layout()
    p2 = save_fig(fig2, 'learning_rate.png')
    print('Saved:', p2)

# 3) Tom tat (JSON) — chen vao bao cao / bang so lieu
m = getattr(train_result, 'metrics', {}) or {}
best_eval = min((e['eval_loss'] for e in log_history if 'eval_loss' in e), default=None)
summary = {
    'model_name': MODEL_NAME,
    'dataset_path': DATASET_PATH,
    'output_dir': OUTPUT_DIR,
    'report_dir': REPORT_DIR,
    'hyperparameters': {
        'num_train_epochs': NUM_TRAIN_EPOCHS,
        'per_device_train_batch_size': TRAIN_BATCH_SIZE,
        'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
        'effective_batch_size': TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
        'learning_rate': LEARNING_RATE,
        'warmup_ratio': WARMUP_RATIO,
        'warmup_steps': WARMUP_STEPS,
        'weight_decay': WEIGHT_DECAY,
        'max_seq_length': MAX_LENGTH,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'eval_steps': EVAL_STEPS if val_ds is not None else None,
        'save_steps': SAVE_STEPS,
        'logging_steps': LOGGING_STEPS,
    },
    'data_split': {
        'total_samples': len(full_ds),
        'train_samples': len(train_ds),
        'val_samples': len(val_ds) if val_ds is not None else 0,
        'test_samples': len(test_ds) if test_ds is not None else 0,
        'val_ratio': VAL_SIZE,
        'test_ratio': TEST_SIZE,
    },
    'training': {
        'global_step': trainer.state.global_step,
        'train_runtime_seconds': m.get('train_runtime'),
        'train_samples_per_second': m.get('train_samples_per_second'),
        'train_steps_per_second': m.get('train_steps_per_second'),
        'train_loss_final': m.get('train_loss'),
        'final_train_loss_logged': train_loss_pts[-1][1] if train_loss_pts else None,
    },
    'evaluation': {
        'best_eval_loss': best_eval,
        'test_metrics': test_metrics if test_metrics is not None else {},
    },
}

out_json = os.path.join(REPORT_DIR, 'training_summary.json')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Saved:', out_json)
print('\n--- training_summary (tom tat) ---')
print(json.dumps(summary, ensure_ascii=False, indent=2))

Saved: /content/qwen25_7b_instruct_lora/training_report/loss_curve.png
Saved: /content/qwen25_7b_instruct_lora/training_report/learning_rate.png
Saved: /content/qwen25_7b_instruct_lora/training_report/training_summary.json

--- training_summary (tom tat) ---
{
  "model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
  "dataset_path": "/content/finetune_qwen257B.jsonl",
  "output_dir": "/content/qwen25_7b_instruct_lora",
  "report_dir": "/content/qwen25_7b_instruct_lora/training_report",
  "hyperparameters": {
    "num_train_epochs": 2,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "effective_batch_size": 8,
    "learning_rate": 0.0002,
    "warmup_ratio": 0.03,
    "warmup_steps": 78,
    "weight_decay": 0.01,
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "eval_steps": 100,
    "save_steps": 100,
    "logging_steps": 10
  },
  "data_split": {
    "total_samples": 11650,
    "train_samples": 10472,
    

In [8]:
# Save LoRA adapter + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'[OK] Saved LoRA adapter to: {OUTPUT_DIR}')

[OK] Saved LoRA adapter to: /content/qwen25_7b_instruct_lora


In [9]:
# Optional: merge LoRA vao base -> model 16-bit (can VRAM khi merge)
import os

if MERGE_AND_SAVE:
    merged_dir = OUTPUT_DIR + '_merged'
    model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
    print(f'[OK] Saved merged model to: {merged_dir}')
else:
    print('MERGE_AND_SAVE=False, skip merge step.')

# -------------------------------
# Export BEST checkpoint (tiện gọi)
# -------------------------------
import shutil
from pathlib import Path

best_ckpt = getattr(getattr(trainer, 'state', None), 'best_model_checkpoint', None)
BEST_DIR = OUTPUT_DIR + '_best'

if best_ckpt:
    src = Path(best_ckpt)
    dst = Path(BEST_DIR)
    dst.mkdir(parents=True, exist_ok=True)

    # Copy adapter + tokenizer + chat template
    keep = [
        'adapter_model.safetensors',
        'adapter_config.json',
        'tokenizer.json',
        'tokenizer_config.json',
        'chat_template.jinja',
        'special_tokens_map.json',
        'added_tokens.json',
    ]

    copied = 0
    for name in keep:
        s = src / name
        if s.exists():
            shutil.copy2(s, dst / name)
            copied += 1

    print('[OK] best_model_checkpoint =', str(src))
    print('[OK] Exported BEST adapter folder to:', str(dst))
    print('[OK] Files copied:', copied)
else:
    print('[WARN] trainer.state.best_model_checkpoint is None (no val / no best). Skip BEST export.')

# -------------------------------
# Đóng gói để download (nhẹ hơn)
# - Zip BEST nếu có
# - Zip full OUTPUT_DIR như cũ
# -------------------------------
import subprocess

if Path(BEST_DIR).exists():
    _bn_best = os.path.basename(BEST_DIR.rstrip('/'))
    subprocess.run(
        f'cd /content && zip -r {_bn_best}.zip {_bn_best}',
        shell=True,
        check=False,
    )
    print(f'[OK] Zipped BEST: /content/{_bn_best}.zip')

_bn = os.path.basename(OUTPUT_DIR.rstrip('/'))
subprocess.run(
    f'cd /content && zip -r {_bn}.zip {_bn}',
    shell=True,
    check=False,
)
print(f'[OK] Zipped FULL: /content/{_bn}.zip')

MERGE_AND_SAVE=False, skip merge step.
[OK] best_model_checkpoint = /content/qwen25_7b_instruct_lora/checkpoint-300
[OK] Exported BEST adapter folder to: /content/qwen25_7b_instruct_lora_best
[OK] Files copied: 5
[OK] Zipped BEST: /content/qwen25_7b_instruct_lora_best.zip
[OK] Zipped FULL: /content/qwen25_7b_instruct_lora.zip
